# RiskLens AI — Credit Default Prediction

**HackRush 2026** | LendingClub loan data | Out-of-time validation (test = 2018)

Pipeline: Clean → Features → LR + LightGBM → Risk Buckets → Policy

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import joblib
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

from app_config.settings import get_settings
from src.scoring.predictor import RiskLensScorer

settings = get_settings()
REPORTS = settings.reports_path

## 1. Dataset Overview

In [ ]:
features_path = settings.feature_store_path / 'features_v1.parquet'
df = pl.read_parquet(features_path)
print(f'Rows: {len(df):,}  |  Columns: {df.width}')
print(f'Years: {sorted(df["issue_year"].unique().to_list())}')

target_by_year = (
    df.group_by('issue_year')
    .agg(pl.len().alias('loans'), pl.col('target').mean().alias('default_rate'))
    .sort('issue_year')
)
target_by_year

## 2. Model Performance (2018 Out-of-Time Test)

In [ ]:
meta = joblib.load(settings.model_store_path / 'model_metadata.joblib')
comparison = json.loads((REPORTS / 'model_comparison.json').read_text())

metrics_df = pd.DataFrame({
    'Logistic Regression': meta['lr_test_metrics'],
    'LightGBM (Champion)': meta['lgb_test_metrics'],
}).T
metrics_df

## 3. Evaluation Charts

In [ ]:
for chart in ['roc_curve.png', 'pr_curve.png', 'shap_importance.png',
              'risk_bucket_default_rates.png', 'portfolio_risk_distribution.png']:
    path = REPORTS / chart
    if path.exists():
        display(Markdown(f'### {chart}'))
        display(Image(filename=str(path)))

## 4. Risk Bucket Validation

In [ ]:
bucket_df = pd.read_csv(REPORTS / 'risk_bucket_default_rates.csv')
bucket_df

## 5. Portfolio Analysis (Expected Loss)

In [ ]:
portfolio = pd.read_csv(REPORTS / 'portfolio_summary.csv')
portfolio

## 6. Sample Loan Scoring (PD → Bucket → Policy)

In [ ]:
scorer = RiskLensScorer(use_shap=False)
sample = RiskLensScorer.load_sample_from_test_set(n=1, issue_year=2018)
result = scorer.score_application(sample)

print(f"Model: {result['model']}")
print(f"PD: {result['probability_of_default']:.1%}")
print(f"Risk tier: {result['risk_bucket']['risk_tier']} ({result['risk_bucket']['credit_grade']})")
print(f"Policy: {result['policy_decision']['decision']} — {result['policy_decision']['comments']}")